In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
import chromadb
from google import genai
from openai import OpenAI
import time

c:\Users\ccass\Documents\2026\Github\LLM-Zoomcamp\Main project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ground_truth_data = [
    {
        "question": "Where can I catch a Largemouth Bass?",
        "expected_answer": "You can catch a Largemouth Bass at the Forest Lake.",
        "expected_category": "fish"
    },
    {
        "question": "What time does the Blue Morpho insect appear?",
        "expected_answer": "The Blue Morpho appears between 6:00AM and 12:00AM.",
        "expected_category": "insect"
    },
    {
        "question": "What ingredients do I need to bake an Apple Pie?",
        "expected_answer": "You need 1 Apple, 1 Wheat, 1 Egg, and 1 Butter.",
        "expected_category": "cook"
    },
    {
        "question": "How much does a 5-star Blueberry Jam cost?",
        "expected_answer": "A 5-star Blueberry Jam costs 1360 coins.",
        "expected_category": "cook"
    },
    {
        "question": "Where can I buy Blue Sugar?",
        "expected_answer": "You can buy Blue Sugar at Doris' shop.",
        "expected_category": "crops"
    },
    {
        "question": "How much does a seed of wheat cost?",
        "expected_answer": "A seed of wheat costs 95 coins.",
        "expected_category": "crops"
    },
    {
        "question": "Where can I buy meat?",
        "expected_answer": "You can buy meat at the Massimo's shop.",
        "expected_category": "crops"
    },
    {
        "question": "Where can I buy seeds of Tea Tree?",
        "expected_answer": "You can buy seeds of Tea Tree at Blanc's shop.",
        "expected_category": "crops"
    },
    {
        "question" : "If I sell 50 1 star truffle pasta, how much raw profit will I make?",
        "expected_answer": "If you sell 50 truffle pasta, you will make a 23950 coins profit.",
        "expected_category": "cook" 
    },
    {
        "question" : "If I sell 10 3 star Blueberry Jam and 5 5 star Blueberry Jam, how much will I make?",
        "expected_answer": "If you sell 10 3 star Blueberry Jam and 5 5 star Blueberry Jam, you will make a total of 10200 coins.",
        "expected_category": "cook"
    },
    {
        "question" : "If I sell 3-star King Eider birds and a 5-star Great Green Macaw, how much will I make?",
        "expected_answer": "If you sell 3-star King Eider birds and a 5-star Great Green Macaw, you will make a total of 620 coins.",
        "expected_category": "bird"
    },
    {
        "question" : "If I sell a 5 star truffle pie and a 3 star Amur Falcon bird, how much will I make?",
        "expected_answer": "If you sell a 5 star truffle pie and a 3 star Amur Falcon bird, you will make a total of 7160 coins.",
        "expected_category": ["cook", "bird"]
    },
    {
        "question" : "If I sell 15 Apollo and 3 Pink Kayatid, how much will I make?",
        "expected_answer": "If you sell 15 Apollo and 3 Pink Kayatid, you will make a total of 720 coins.",
        "expected_category": "insect"
    },
    {
        "question" : "Can I buy Tiramisu from Blanc?",
        "expected_answer": "No, you cannot buy Tiramisu from Blanc.",
        "expected_category": "crops"
    },
    {
        "question" : "Can I buy Tiramisu from Massimo?",
        "expected_answer": "No, you cannot buy Tiramisu from Massimo.",
        "expected_category": "crops"
    },
    {
        "question" : "Where can I buy clothes?",
        "expected_answer": "I don't have data for that.",
        "expected_category": "none"
    },
    {
        "question" : "Where can I buy a 5-star Blueberry Jam?",
        "expected_answer": "You cannot buy a 5-star Blueberry Jam from any shop.",
        "expected_category": "cook"
    },
    {
        "question" : "Where can I find a wood pecker bird?",
        "expected_answer": "I dont have any data for that.",
        "expected_category": "none"
    }

]

In [6]:
load_dotenv(override=True)

import chromadb
client = chromadb.HttpClient(host='localhost', port=8000)
collection = client.get_collection(name="heartopia_knowledge")

model = SentenceTransformer("all-MiniLM-L6-v2")

gemini_client = genai.Client()
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9279.04it/s]


In [3]:
def retrieve_docs(user_query: str, top_k: int = 5):
    """Retrieves document chunks and metadata from ChromaDB."""
    query_vector = model.encode(user_query).tolist()
    results = collection.query(
        query_embeddings=[query_vector],
        n_results=top_k
    )
    docs = results['documents'][0]
    metadatas = results['metadatas'][0] if 'metadatas' in results else []
    return docs, metadatas

def generate_rag_response(prompt_text: str, max_retries: int = 3):
    """Generates an answer using Gemini, retrying if the server is overloaded."""
    gemini_api_key = os.getenv("GEMINI_API_KEY")
    if not gemini_api_key:
        return "Generation Failed: GEMINI_API_KEY is missing from .env!"
        
    client = genai.Client(api_key=gemini_api_key)
    
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model='gemini-3.5-flash',
                contents=prompt_text
            )
            return response.text
            
        except Exception as gemini_error:
            error_str = str(gemini_error)
            print(f"\n Gemini Attempt {attempt + 1} Failed: {error_str}")

            if "503" in error_str or "UNAVAILABLE" in error_str:
                if attempt < max_retries - 1:
                    wait_time = 60 * (attempt + 1)
                    time.sleep(wait_time)
                    continue
            
            try:
                openai_response = openai_client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=[
                        {"role": "system", "content": "You are a helpful Heartopia game assistant and financial advisor."},
                        {"role": "user", "content": prompt_text}
                    ]
                )
                return openai_response.choices[0].message.content
            except Exception as openai_error:
                return f"Generation Failed - Gemini: {gemini_error} | OpenAI: {openai_error}"

In [5]:
import time

eval_results = []

for idx, test_case in enumerate(ground_truth_data, 1):
    question = test_case["question"]
    expected_ans = test_case["expected_answer"]
    expected_cat = test_case["expected_category"]
    
    docs, metadatas = retrieve_docs(question, top_k=5)
    
    retrieved_categories = [m.get("category", "") for m in metadatas if isinstance(m, dict)]
    
    if expected_cat == "none":
        hit = True
    elif isinstance(expected_cat, list):
        hit = any(cat in retrieved_categories for cat in expected_cat)
    else:
        hit = expected_cat in retrieved_categories

    full_context = "\n".join(docs)
    prompt = f"""You are an expert Heartopia game assistant and financial advisor. 
Use ONLY the following Context to answer the User's question. You are highly encouraged to perform step-by-step mathematical calculations if needed. Do not hallucinate prices, ingredients, or locations. If the answer is not explicitly contained within the Context below, respond with "I don't have data for that." If the user prompt does not explicitly state star rate, assume it to be 1 star value.

Context:
{full_context}

User Question: {question}
"""

    llm_answer = generate_rag_response(prompt)
    
    eval_results.append({
        "ID": idx,
        "Question": question,
        "Expected Category": str(expected_cat),
        "Retrieval Hit": hit,
        "Expected Answer": expected_ans,
        "LLM Answer": llm_answer
    })
    print(f"[{idx}/{len(ground_truth_data)}] Evaluated: '{question}' | Retrieval Hit: {hit}")

    time.sleep(60)

df_eval = pd.DataFrame(eval_results)
hit_rate = (df_eval["Retrieval Hit"].sum() / len(df_eval)) * 100

print("\n" + "="*50)
print(f"SUMMARY: Retrieval Hit Rate = {hit_rate:.2f}%")
print("="*50)

df_eval.to_csv("evaluation_results.csv", index=False)

[1/18] Evaluated: 'Where can I catch a Largemouth Bass?' | Retrieval Hit: True
[2/18] Evaluated: 'What time does the Blue Morpho insect appear?' | Retrieval Hit: True
[3/18] Evaluated: 'What ingredients do I need to bake an Apple Pie?' | Retrieval Hit: True
[4/18] Evaluated: 'How much does a 5-star Blueberry Jam cost?' | Retrieval Hit: True
[5/18] Evaluated: 'Where can I buy Blue Sugar?' | Retrieval Hit: True
[6/18] Evaluated: 'How much does a seed of wheat cost?' | Retrieval Hit: True
[7/18] Evaluated: 'Where can I buy meat?' | Retrieval Hit: False
[8/18] Evaluated: 'Where can I buy seeds of Tea Tree?' | Retrieval Hit: True
[9/18] Evaluated: 'If I sell 50 1 star truffle pasta, how much raw profit will I make?' | Retrieval Hit: True
[10/18] Evaluated: 'If I sell 10 3 star Blueberry Jam and 5 5 star Blueberry Jam, how much will I make?' | Retrieval Hit: True
[11/18] Evaluated: 'If I sell 3-star King Eider birds and a 5-star Great Green Macaw, how much will I make?' | Retrieval Hit: True

Evaluation: LLM as Judge

In [4]:
from google import genai
import os
from dotenv import load_dotenv
from google.genai import types

load_dotenv()
client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

In [5]:
def retrieve_and_generate(user_query: str):

    docs, metadatas = retrieve_docs(user_query, top_k=5)

    context_str = "\n".join(docs)
    prompt = f"""
    You are a helpful Heartopia game assistant and financial advisor.
    Use ONLY the context provided below to answer the user's question.
    If the answer cannot be found in the context, state "I don't have data for that."

    Context:
    {context_str}

    Question: {user_query}
    """

    generated_answer = generate_rag_response(prompt)

    return docs, generated_answer

In [6]:
def evaluate_with_llm_judge(question, context, generated_answer, expected_answer):
    judge_prompt = f"""
    You are an expert AI evaluator judging an answer generated by a RAG assistant for the game Heartopia.

    User Question: {question}
    Retrieved Context from Database: {context}
    Expected Ground Truth Answer: {expected_answer}
    Generated Assistant Answer: {generated_answer}

    Please rate the Generated Assistant Answer on two metrics from 1 to 5:

    1. **Faithfulness** (1-5): Does the generated answer rely ONLY on the retrieved context? (Give 5 if it strictly uses the context without hallucinating extra details. Give 1 if it invents information not in the context).
    2. **Relevance** (1-5): Does the generated answer directly address the user's question? (Give 5 if it fully and accurately answers the question. Give 1 if it ignores the question or gives an irrelevant response).

    Output strictly valid JSON in this format:
    {{
        "faithfulness": <number between 1 and 5>,
        "relevance": <number between 1 and 5>,
        "reasoning": "<short sentence explaining the score>"
    }}
    """

    response = judge_client.models.generate_content(
        model="gemini-3.5-flash", contents=judge_prompt
    )

    cleaned_text = (
        response.text.replace("```json", "").replace("```", "").strip()
    )

    try:
        return json.loads(cleaned_text)
    except Exception as e:
        return {
            "faithfulness": None,
            "relevance": None,
            "reasoning": f"Parsing error: {e}",
        }

In [7]:
print("ground_truth_data" in globals())

True


In [ ]:
import time
import pandas as pd
import json
import os
from pathlib import Path
from google import genai
from sentence_transformers import SentenceTransformer

judge_client = genai.Client()

ground_path = Path("data/ground_truth.json")
if not ground_path.exists():
    ground_truth_data = ground_truth_data 
else:
    ground_truth_data = json.loads(ground_path.read_text())

results = []
for item in ground_truth_data:
    q = item["question"]
    expected = item["expected_answer"]

    if "model" not in globals():
        model = SentenceTransformer("all-MiniLM-L6-v2")

    retrieved_docs, metadatas = retrieve_docs(q, top_k=5)
    
    context_for_prompt = "\n".join([doc if isinstance(doc, str) else str(doc) for doc in retrieved_docs])
    prompt = f"""
    You are a helpful Heartopia game assistant and financial advisor.
    Use ONLY the context provided below to answer the user's question.
    If the answer cannot be found in the context, state "I don't have data for that."

    Context:
    {context_for_prompt}

    Question: {q}
    """
    generated_answer = generate_rag_response(prompt)

    scores = evaluate_with_llm_judge(q, context_for_prompt, generated_answer, expected)

    results.append(
        {
            "question": q,
            "expected_answer": expected,
            "generated_answer": generated_answer,
            "faithfulness": scores.get("faithfulness"),
            "relevance": scores.get("relevance"),
            "reasoning": scores.get("reasoning"),
        }
    )

    print(
        f"Q: {q[:35]}... | Faithfulness: {scores.get('faithfulness')} | Relevance: {scores.get('relevance')}"
    )

    time.sleep(300)

eval_df = pd.DataFrame(results)
print("\n Evaluation Complete!")

Q: Where can I catch a Largemouth Bass... | Faithfulness: 5 | Relevance: 5
Q: What time does the Blue Morpho inse... | Faithfulness: 5 | Relevance: 5
Q: What ingredients do I need to bake ... | Faithfulness: 5 | Relevance: 5
Q: How much does a 5-star Blueberry Ja... | Faithfulness: 5 | Relevance: 5
Q: Where can I buy Blue Sugar?... | Faithfulness: 5 | Relevance: 5
Q: How much does a seed of wheat cost?... | Faithfulness: 5 | Relevance: 5
Q: Where can I buy meat?... | Faithfulness: 5 | Relevance: 5
Q: Where can I buy seeds of Tea Tree?... | Faithfulness: 5 | Relevance: 5


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 755.976017ms.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.5-flash'}, 

In [ ]:
eval_df.to_csv("generation_evaluation_results.csv", index=False)

avg_faithfulness = eval_df["faithfulness"].mean()
avg_relevance = eval_df["relevance"].mean()

print(f"Average Faithfulness Score: {avg_faithfulness:.2f} / 5.0")
print(f"Average Relevance Score:    {avg_relevance:.2f} / 5.0")

eval_df[["question", "faithfulness", "relevance", "reasoning"]].head()